In [22]:
"""
Stream_demo.py

Demonstrates the full streaming ML pipeline:
1. Load data (simulate with random)
2. Split into chunks
3. Train incrementally with partial_fit
4. Log metrics and visualise
"""

'\nStream_demo.py\n\nDemonstrates the full streaming ML pipeline:\n1. Load data (simulate with random)\n2. Split into chunks\n3. Train incrementally with partial_fit\n4. Log metrics and visualise\n'

In [23]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Added:", project_root)

Added: /Users/ayushsahane/Documents/PR_AIML/numcompute-stream


In [24]:
import numpy as np
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.preprocessing import StandardScaler, Imputer
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.metrics import Accuracy, ConfusionMatrix
from numcompute_stream.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth,
    plot_confusion_matrix,
)

In [25]:
def main():
    print("=" * 70)
    print("NumCompute-Stream: Stream Demo")
    print("=" * 70)

    # 1. Generate synthetic streaming data
    print("\n[1] Generating synthetic data...")
    rng = np.random.default_rng(42)
    n_samples_total = 2000
    n_features = 5
    n_chunks = 8
    chunk_size = n_samples_total // n_chunks

    X = rng.normal(loc=0.5, scale=2.0, size=(n_samples_total, n_features))
    # Add some NaNs to test robustness
    nan_mask = rng.random((n_samples_total, n_features)) < 0.05
    X[nan_mask] = np.nan

    # Binary classification: based on first feature
    y = (X[:, 0] > 0.5).astype(int)

    print(f"   Total samples: {n_samples_total}")
    print(f"   Features: {n_features}")
    print(f"   Chunks: {n_chunks}")
    print(f"   Chunk size: {chunk_size}")

    # 2. Build pipeline
    print("\n[2] Building streaming pipeline...")
    pipe_single = Pipeline([
        ("impute", Imputer()),
        ("scale", StandardScaler()),
        ("model", EnsembleClassifier(n_estimators=3, max_depth=4, random_state=0)),
    ])

    pipe_ensemble = Pipeline([
        ("impute", Imputer()),
        ("scale", StandardScaler()),
        ("model", EnsembleClassifier(n_estimators=5, max_depth=4, random_state=1)),
    ])

    # 3. Train on chunks
    print("\n[3] Training on chunks...")

    trainer_single = StreamTrainer(pipe_single, metric=Accuracy())
    trainer_ensemble = StreamTrainer(pipe_ensemble, metric=Accuracy())

    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        end = start + chunk_size
        X_chunk = X[start:end]
        y_chunk = y[start:end]

        info_s = trainer_single.fit_chunk(X_chunk, y_chunk)
        info_e = trainer_ensemble.fit_chunk(X_chunk, y_chunk)

        print(f"   Chunk {chunk_idx + 1}/{n_chunks}: "
              f"Single Acc={info_s['cumulative_accuracy']:.3f}, "
              f"Ensemble Acc={info_e['cumulative_accuracy']:.3f}")

    # 4. Extract metrics 
    print("\n[4] Extracting metrics...")
    acc_single = [h["cumulative_accuracy"] for h in trainer_single.history_]
    acc_ensemble = [h["cumulative_accuracy"] for h in trainer_ensemble.history_]
    chunk_acc_single = [h["chunk_accuracy"] for h in trainer_single.history_]

    # 5. Visualise
    print("\n[5] Visualising results...")

    print("   - Plotting single model accuracy over time...")
    plot_metric_over_time(
        acc_single,
        title="Single Ensemble: Cumulative Accuracy Over Chunks",
        ylabel="Accuracy",
        save_path="plots/single_accuracy.png",
        show=False,
    )

    print("   - Comparing two ensemble models...")
    compare_models(
        acc_single,
        acc_ensemble,
        labels=("3-tree Ensemble", "5-tree Ensemble"),
        title="Ensemble Size Comparison",
        ylabel="Cumulative Accuracy",
        save_path="plots/ensemble_comparison.png",
        show=False,
    )

    print("   - Plotting chunk accuracy...")
    plot_metric_over_time(
        chunk_acc_single,
        title="Per-Chunk Accuracy",
        ylabel="Chunk Accuracy",
        save_path="plots/chunk_accuracy.png",
        show=False,
    )

    # 6. Final evaluation 
    print("\n[6] Final evaluation...")
    y_pred_single = pipe_single.predict(X)
    y_pred_ensemble = pipe_ensemble.predict(X)

    final_acc_single = np.mean(y_pred_single == y)
    final_acc_ensemble = np.mean(y_pred_ensemble == y)

    print(f"   Single Ensemble Final Accuracy: {final_acc_single:.4f}")
    print(f"   Multi Ensemble Final Accuracy: {final_acc_ensemble:.4f}")

    # Confusion matrices
    cm_single = ConfusionMatrix(n_classes=2)
    cm_ensemble = ConfusionMatrix(n_classes=2)

    cm_single.update(y, y_pred_single)
    cm_ensemble.update(y, y_pred_ensemble)

    print("\n   Confusion Matrix (Single):")
    print(cm_single.result())
    print("\n   Confusion Matrix (Ensemble):")
    print(cm_ensemble.result())

    print("\n" + "=" * 70)
    print("Demo complete! Check 'plots/' folder for visualisations.")
    print("=" * 70)

In [26]:
import os

os.makedirs("plots", exist_ok=True)

print("Plots folder created")
if __name__ == "__main__":
    main()

Plots folder created
NumCompute-Stream: Stream Demo

[1] Generating synthetic data...
   Total samples: 2000
   Features: 5
   Chunks: 8
   Chunk size: 250

[2] Building streaming pipeline...

[3] Training on chunks...
   Chunk 1/8: Single Acc=1.000, Ensemble Acc=1.000
   Chunk 2/8: Single Acc=0.998, Ensemble Acc=1.000
   Chunk 3/8: Single Acc=0.999, Ensemble Acc=1.000
   Chunk 4/8: Single Acc=0.998, Ensemble Acc=1.000
   Chunk 5/8: Single Acc=0.998, Ensemble Acc=1.000
   Chunk 6/8: Single Acc=0.999, Ensemble Acc=1.000
   Chunk 7/8: Single Acc=0.998, Ensemble Acc=0.999
   Chunk 8/8: Single Acc=0.999, Ensemble Acc=1.000

[4] Extracting metrics...

[5] Visualising results...
   - Plotting single model accuracy over time...
Plot saved to plots/single_accuracy.png
   - Comparing two ensemble models...
Plot saved to plots/ensemble_comparison.png
   - Plotting chunk accuracy...
Plot saved to plots/chunk_accuracy.png

[6] Final evaluation...
   Single Ensemble Final Accuracy: 0.9975
   Multi 